### Libraries

In [ ]:
import os, re, json 
from pathlib import Path
import pandas as pd
import re, json, unicodedata, pdfplumber
from pathlib import Path

## Regulars Expresion

In [27]:
# Nº Cámara  →    298/2022C   ó 298/2022C 
RE_CAMARA = re.compile(r"(\d+)\s*/\s*(\d{4})\s*C?", re.I)
# Nº Senado  →    014/2022S
RE_SENADO = re.compile(r"(\d+)\s*/\s*(\d{4})\s*S",  re.I)
# Final estándar “PROYECTO DE …”
RE_TIPO   = re.compile(
        r"PROYECTO\s+DE\s+(?:ACTO\s+LEGISLATIVO|LEY(?:\s+\w+)?)\s*$",
        re.I
    )

In [28]:
def norm(s: str) -> str:
    """Normaliza espacios y elimina saltos de línea."""
    if s is None:
        return ""
    s = unicodedata.normalize("NFKC", s).strip()
    return re.sub(r"\s+", " ", s)

In [29]:
def parse_num(cadena: str, patron: re.Pattern):
    """Devuelve (numero, año) o (None, None)."""
    m = patron.search(cadena)
    return (m.group(1), m.group(2)) if m else (None, None)

In [30]:
def split_pseud_titulo_tipo(cad: str):
    """Devuelve (pseudónimo, título, tipo_ley) a partir de la cadena completa."""
    m_tipo = RE_TIPO.search(cad)
    if m_tipo:
        tipo  = m_tipo.group(0).strip()
        resto = cad[:m_tipo.start()].strip()
    else:
        tipo, resto = "", cad.strip()

    m_pseud = re.match(r"(.+?)\s+Por\b", resto, re.I)
    if m_pseud:
        pseud  = m_pseud.group(1).strip()
        titulo = resto[m_pseud.end()-3:].strip()   # deja «Por …»
    else:
        pseud, titulo = resto, ""
    return pseud, titulo, tipo


## Process for page

In [ ]:
def extraer_filas_pdf(path_pdf: str) -> list[dict]:
    filas = []
    with pdfplumber.open(path_pdf) as pdf:
        for pag in pdf.pages:
            for tabla in pag.extract_tables():
                cab = [norm(c) for c in tabla[0]]
                if cab[:2] not in (["CANT.", "No CÁMARA"], ["CANT", "No CÁMARA"]):
                    continue

                for raw in tabla[1:]:
                    raw = (raw + [""] * 6)[:6]          # siempre 6 columnas
                    cant, camara, senado, pseud, titulo, tipo_ley = map(norm, raw)

                    if not camara and not pseud and not titulo and not tipo_ley:
                        continue

                    # 1️⃣ Senado incrustado en «pseud»
                    if not senado and RE_SENADO.match(pseud):
                        senado, pseud = pseud, ""
                    # ------------------------------------------------------------------
                    # 2️⃣ Bloque de corrección de columnas fusionadas *******************
                    #    (casos A, B, C explicados anteriormente)
                    # ------------------------------------------------------------------
                    if not titulo and not tipo_ley and "PROYECTO" in pseud.upper():
                        pseud, titulo, tipo_ley = split_pseud_titulo_tipo(pseud)

                    elif titulo and not tipo_ley and "PROYECTO" in titulo.upper():
                        joined = f"{pseud} {titulo}".strip()
                        pseud, titulo, tipo_ley = split_pseud_titulo_tipo(joined)

                    elif not titulo and not pseud and tipo_ley:
                        pseud, titulo, tipo_ley = split_pseud_titulo_tipo(tipo_ley)
                    # ------------------------------------------------------------------

                    # Limpia numeración pegada al pseudónimo
                    pseud = re.sub(r"^\d+\s+", "", pseud)

                    numC, anioC = parse_num(camara, RE_CAMARA)
                    numS, anioS = parse_num(senado, RE_SENADO)
 
                    filas.append({
                        "numeroCamara": numC,
                        "anioCamara":   anioC,
                        "numeroSenado": numS,
                        "anioSenado":   anioS,
                        "pseudonimo":   pseud,
                        "titulo":       titulo,
                        "tipoLey":      tipo_ley,
                    })
    return filas 

## function principal  to pdf

In [32]:
def procesar_pdf_individual(path_pdf: str, carpeta_salida: str):
    path_pdf        = Path(path_pdf)
    carpeta_salida  = Path(carpeta_salida)
    carpeta_salida.mkdir(parents=True, exist_ok=True)

    filas = extraer_filas_pdf(path_pdf)
    print(f"✅ Total de filas extraídas: {len(filas)}")

    if not filas:
        print("⚠️  No se detectaron filas con metadatos válidos.")
        return

    base = path_pdf.stem
    for i, fila in enumerate(filas, 1):
        with open(carpeta_salida / f"{base}_{i:04d}.json",
                  "w", encoding="utf-8") as f:
            json.dump(fila, f, indent=4, ensure_ascii=False)

    print(f"📁 JSON guardados en: {carpeta_salida.resolve()}")

In [33]:
if __name__ == "__main__":
    pdf_entrada   = r"C:\Users\juans\Documents\pro\Model-Extract-information\document\resource\comisiones\2022 2023 LEGISLATURA_comision_1.pdf"
    carpeta_json  = r"C:\Users\juans\Documents\pro\Model-Extract-information\document\2022_2023\comision\comision_2"

    procesar_pdf_individual(pdf_entrada, carpeta_json)

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


✅ Total de filas extraídas: 146
📁 JSON guardados en: C:\Users\juans\Documents\pro\Model-Extract-information\document\2022_2023\comision\comision_2
